In [7]:
# Data manipulation
import pandas as pd
import numpy as np
from collections import Counter

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

# Preprocessing
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate, RandomizedSearchCV
from imblearn.over_sampling import SMOTE

# XGBoost
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                            roc_auc_score, roc_curve, confusion_matrix,
                            classification_report, precision_recall_curve)
from scipy.stats import randint, uniform

# For comparison with other models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

We have already done the EDA.

## Data Splitting & Preprocessing

Split first, then scale. We use **RobustScaler** (instead of StandardScaler) because the dataset is heavily imbalanced — outliers could distort the scaling and hide true fraud patterns.

We then apply **SMOTE only on the training set** to avoid data leakage.

In [8]:
df = pd.read_csv('creditcard.csv')

X = df.drop('Class', axis=1)
y = df['Class']

# 1. Split FIRST (before any transformation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Scale features (fit on training only)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print("Class distribution after SMOTE:")
print(y_train_balanced.value_counts())

Class distribution after SMOTE:
Class
0    227451
1    227451
Name: count, dtype: int64


## Optimization: Fix Double Imbalance Handling

**Original issue:** The previous version used both SMOTE *and* `scale_pos_weight=577` simultaneously.
This is a form of **double-counting** — SMOTE already balances the classes to 50/50,
so applying a large `scale_pos_weight` on top further over-penalizes non-fraud predictions.
This is why Precision was only 59% while Recall was pushed very high.

**Fix:** Since SMOTE has already balanced the training set, we set `scale_pos_weight=1`.
The model now learns from a balanced dataset without any additional penalty skew.

**Expected effect:** Precision improves, Recall may decrease slightly — this is the deliberate trade-off.

In [9]:
# Define parameter grid
param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': randint(1, 7),
    'gamma': uniform(0, 0.5)
}

# OPTIMIZATION: scale_pos_weight=1 because SMOTE already balanced the training set.
# Using scale_pos_weight=577 on top of SMOTE was double-penalizing non-fraud,
# which inflated Recall but hurt Precision significantly.
model = XGBClassifier(
    scale_pos_weight=1,  # <-- Changed from 577 to 1
    eval_metric='logloss',
    random_state=42,
    use_label_encoder=False
)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=50,
    scoring='f1',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train_balanced, y_train_balanced)

print("Best parameters:", random_search.best_params_)
print("Best F1 score:", random_search.best_score_)

best_model = random_search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters: {'colsample_bytree': 0.9141362604455774, 'gamma': 0.3344941273571143, 'learning_rate': 0.1842059864309364, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 198, 'subsample': 0.989465534702127}
Best F1 score: 0.9997934070276701


In [10]:
# Evaluate on test set
y_pred = best_model.predict(X_test_scaled)
y_pred_proba = best_model.predict_proba(X_test_scaled)[:, 1]

print("=" * 50)
print("TEST SET PERFORMANCE (Optimized)")
print("=" * 50)
print(classification_report(y_test, y_pred))
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")

TEST SET PERFORMANCE (Optimized)
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.77      0.86      0.81        98

    accuracy                           1.00     56962
   macro avg       0.89      0.93      0.91     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC:   0.9812
Recall:    0.8571
Precision: 0.7706
F1 Score:  0.8116


## Bonus: Threshold Tuning

Beyond fixing `scale_pos_weight`, we can further control the Precision/Recall trade-off
by adjusting the **decision threshold** (default = 0.5).

- **Higher threshold** → fewer fraud predictions → higher Precision, lower Recall
- **Lower threshold** → more fraud predictions → lower Precision, higher Recall

This is useful when you want to tune the model for a specific business requirement
without retraining.

In [11]:
print("Threshold Tuning — Precision vs Recall Trade-off")
print("-" * 55)
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 55)

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
for t in thresholds:
    y_pred_t = (y_pred_proba >= t).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    f = f1_score(y_test, y_pred_t, zero_division=0)
    print(f"{t:>10.1f} {p:>10.3f} {r:>10.3f} {f:>10.3f}")

Threshold Tuning — Precision vs Recall Trade-off
-------------------------------------------------------
 Threshold  Precision     Recall         F1
-------------------------------------------------------
       0.3      0.696      0.888      0.780
       0.4      0.752      0.867      0.806
       0.5      0.771      0.857      0.812
       0.6      0.806      0.847      0.826
       0.7      0.838      0.847      0.843
       0.8      0.883      0.847      0.865


## Metric Reference

| Metric | Formula | Optimizes | Use When |
|--------|---------|-----------|----------|
| Precision | TP / (TP + FP) | Minimize false positives | False alarms are expensive |
| Recall | TP / (TP + FN) | Minimize false negatives | Missing fraud is dangerous |
| F1 | 2 × P × R / (P + R) | Balance both | Equal costs or single metric needed |
| ROC-AUC | Area under ROC curve | Overall discrimination | Compare models regardless of threshold |

**In fraud detection:** Recall is typically prioritized because the cost of missing a real fraud
far exceeds the cost of a false alarm. However, precision matters too — too many false alarms
overload the review team and erode trust in the system.